# MalthusJAX Level 1: Genome Primitives Demo

This notebook demonstrates the **core genome types** in MalthusJAX:
- `BinaryGenome` — for combinatorial/discrete optimization
- `RealGenome` — for continuous optimization

Key concepts covered:
1. Configuration classes and the `shape`/`length` API
2. Random initialization with JAX PRNG
3. Population containers (SoA pattern)
4. Distance metrics and utility methods
5. JAX integration (`jax.vmap`, `jax.jit`)

In [1]:
# Core imports
import jax
import jax.numpy as jnp
import jax.random as jr

# MalthusJAX genome imports
from malthusjax.core.genome import (
    BinaryGenome,
    BinaryGenomeConfig,
    BinaryPopulation,
    RealGenome,
    RealGenomeConfig,
    RealPopulation,
)

# Reproducibility
key = jr.PRNGKey(42)
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

JAX version: 0.8.0
Devices: [CpuDevice(id=0)]


---
## 1. Binary Genome

Binary genomes represent solutions as bit-strings (0/1 sequences). Common use cases:
- Knapsack problem
- Feature selection
- Boolean satisfiability (SAT)

In [2]:
# Create a config using the modern `shape` API
binary_config = BinaryGenomeConfig(shape=(20,), p=0.5)
print(f"Config shape: {binary_config.shape}")
print(f"Resolved shape: {binary_config.resolved_shape}")

# Backwards-compatible: using the legacy `length` alias
legacy_config = BinaryGenomeConfig(length=20, p=0.5)
print(f"\nLegacy config length: {legacy_config.length}")
print(f"Legacy resolved shape: {legacy_config.resolved_shape}")

Config shape: (20,)
Resolved shape: (20,)

Legacy config length: 20
Legacy resolved shape: (20,)


In [3]:
# Initialize a single binary genome
key, subkey = jr.split(key)
genome = BinaryGenome.random_init(subkey, binary_config)

print(f"Genome: {genome}")
print(f"Shape: {genome.shape}")
print(f"Size: {genome.size}")
print(f"Values: {genome.values}")

Genome: <BinaryGenome(0011111111..., len=20)>
Shape: (20,)
Size: 20
Values: [0 0 1 1 1 1 1 1 1 1 1 1 0 0 0 1 1 0 1 1]


In [4]:
# Binary genome utilities
print(f"Count of 1s (Hamming weight): {genome.count_ones()}")
print(f"Integer value (MSB first): {genome.to_int(msb_first=True)}")
print(f"Integer value (LSB first): {genome.to_int(msb_first=False)}")

# Flip a bit (functional update)
flipped = genome.flip_bit(0)
print(f"\nOriginal bit 0: {genome.values[0]}")
print(f"Flipped bit 0:  {flipped.values[0]}")

Count of 1s (Hamming weight): 14
Integer value (MSB first): 261915
Integer value (LSB first): 888828

Original bit 0: 0
Flipped bit 0:  1


---
## 2. Real Genome

Real genomes represent solutions as continuous vectors in $\mathbb{R}^n$. Common use cases:
- Function optimization (Sphere, Rastrigin, etc.)
- Neural network weight evolution
- Parameter estimation

In [5]:
# Real genome config with bounds
real_config = RealGenomeConfig(
    shape=(10,),
    bounds=(-5.0, 5.0),
    dtype=jnp.float32,
)

key, subkey = jr.split(key)
real_genome = RealGenome.random_init(subkey, real_config)

print(f"Real genome shape: {real_genome.shape}")
print(f"Values: {real_genome.values}")
print(f"Magnitude (L2 norm): {real_genome.magnitude():.4f}")

Real genome shape: (10,)
Values: [-0.8351362  -4.1352644  -0.17947912 -3.7567604   4.7219715  -1.8332362
  2.414676    2.5007951   4.9841557  -3.0907142 ]
Magnitude (L2 norm): 10.2020


In [6]:
# Autocorrect enforces bounds
# Manually create an out-of-bounds genome
oob_genome = RealGenome(values=jnp.array([10.0, -10.0, 3.0, 0.0, 100.0]))
print(f"Out-of-bounds values: {oob_genome.values}")

config_for_clip = RealGenomeConfig(shape=(5,), bounds=(-5.0, 5.0))
corrected = oob_genome.autocorrect(config_for_clip)
print(f"After autocorrect:    {corrected.values}")

Out-of-bounds values: [ 10. -10.   3.   0. 100.]
After autocorrect:    [ 5. -5.  3.  0.  5.]


---
## 3. Distance Metrics

Both genome types support distance calculations for diversity analysis.

In [7]:
# Binary distance (Hamming = count of differing bits)
key, k1, k2 = jr.split(key, 3)
g1 = BinaryGenome.random_init(k1, binary_config)
g2 = BinaryGenome.random_init(k2, binary_config)

hamming_dist = g1.distance(g2, metric="hamming")
euclidean_dist = g1.distance(g2, metric="euclidean")

print("Binary genomes:")
print(f"  g1: {g1.values[:10]}...")
print(f"  g2: {g2.values[:10]}...")
print(f"  Hamming distance:   {hamming_dist}")
print(f"  Euclidean distance: {euclidean_dist:.4f}")

Binary genomes:
  g1: [1 1 1 1 0 0 1 0 0 0]...
  g2: [0 0 0 0 0 0 0 0 0 1]...
  Hamming distance:   10
  Euclidean distance: 3.1623


In [8]:
# Real distance (Euclidean, Manhattan)
key, k1, k2 = jr.split(key, 3)
r1 = RealGenome.random_init(k1, real_config)
r2 = RealGenome.random_init(k2, real_config)

print("Real genomes:")
print(f"  r1: {r1.values[:5]}...")
print(f"  r2: {r2.values[:5]}...")
print(f"  Euclidean (L2): {r1.distance(r2, 'euclidean'):.4f}")
print(f"  Manhattan (L1): {r1.distance(r2, 'manhattan'):.4f}")

Real genomes:
  r1: [ 2.4964988 -1.9459963 -1.1179674  4.788227   2.2960818]...
  r2: [-2.132901   -0.84871054  3.1789303  -0.18130064 -3.6772716 ]...
  Euclidean (L2): 17.1024
  Manhattan (L1): 46.3684


---
## 4. Populations (Struct-of-Arrays)

Populations follow the **SoA pattern**: a single `genes` object where each leaf array has a leading population dimension `N`.

In [9]:
# Create a population of binary genomes
POP_SIZE = 50
key, subkey = jr.split(key)

binary_pop = BinaryPopulation.init_random(subkey, binary_config, size=POP_SIZE)

print(f"Population size: {len(binary_pop)}")
print(f"Genes shape: {binary_pop.genes.values.shape}")
print(f"Fitness shape: {binary_pop.fitness.shape}")
print(f"First individual: {binary_pop[0]}")

Population size: 50
Genes shape: (50, 20)
Fitness shape: (50,)
First individual: <BinaryGenome(1010001010..., len=20)>


In [10]:
# Create a population of real genomes
key, subkey = jr.split(key)
real_pop = RealPopulation.init_random(subkey, real_config, size=POP_SIZE)

print(f"Real population size: {len(real_pop)}")
print(f"Genes shape: {real_pop.genes.values.shape}")

Real population size: 50
Genes shape: (50, 10)


In [11]:
# Population slicing
sub_pop = binary_pop[10:20]  # Slice returns a new population
print(f"Sub-population size: {len(sub_pop)}")

individual = binary_pop[0]  # Integer index returns a single genome
print(f"Individual type: {type(individual).__name__}")

Sub-population size: 10
Individual type: BinaryGenome


In [12]:
# Distance matrix (pairwise distances)
small_pop = RealPopulation.init_random(jr.PRNGKey(99), real_config, size=5)
dist_matrix = small_pop.distance_matrix(metric="euclidean")

print("Pairwise distance matrix (5x5):")
print(dist_matrix)

Pairwise distance matrix (5x5):
[[ 0.       13.565277 14.653974 13.275057 15.035684]
 [13.565277  0.        9.993956 12.348368 13.859613]
 [14.653974  9.993956  0.       10.687743 11.912414]
 [13.275057 12.348368 10.687743  0.        5.2059  ]
 [15.035684 13.859613 11.912414  5.2059    0.      ]]


---
## 5. JAX Integration (vmap, jit)

Genomes are **immutable PyTrees**, making them fully compatible with JAX transformations.

In [13]:
# Batch initialization with vmap (equivalent to create_population)
keys = jr.split(jr.PRNGKey(123), 10)
batched_genomes = jax.vmap(BinaryGenome.random_init, in_axes=(0, None))(keys, binary_config)

print(f"Batched genomes shape: {batched_genomes.values.shape}")

Batched genomes shape: (10, 20)


In [14]:
# JIT-compiled fitness evaluation
@jax.jit
def evaluate_binary_sum(genome: BinaryGenome) -> jnp.ndarray:
    """Simple fitness: count of 1s (OneMax problem)."""
    return genome.count_ones()

# Vectorize over population
fitness_fn = jax.vmap(evaluate_binary_sum)
fitness_values = fitness_fn(binary_pop.genes)

print(f"Fitness values (first 10): {fitness_values[:10]}")
print(f"Best fitness: {jnp.max(fitness_values)} / {binary_config.resolved_shape[0]}")

Fitness values (first 10): [10  8 12 10 12 11  8  8  7 16]
Best fitness: 16 / 20


In [15]:
# JIT-compiled Sphere function for real genomes
@jax.jit
def sphere(genome: RealGenome) -> jnp.ndarray:
    """Sphere function: f(x) = sum(x^2). Minimum at origin."""
    return jnp.sum(genome.values ** 2)

sphere_fitness = jax.vmap(sphere)(real_pop.genes)
print(f"Sphere fitness (first 10): {sphere_fitness[:10]}")
print(f"Best (lowest) fitness: {jnp.min(sphere_fitness):.4f}")

Sphere fitness (first 10): [ 94.268105  61.03359   64.40614   91.713684  96.35134   82.24067
  67.54219   94.46992  114.969955  94.78969 ]
Best (lowest) fitness: 27.2015


---
## 6. Spawn Offspring Pattern

Use `spawn_offspring()` to create new populations from modified genes while resetting fitness.

In [16]:
# Simulate mutation: add noise to real genomes
def add_noise(key, genome, sigma=0.1):
    noise = jr.normal(key, genome.values.shape) * sigma
    new_values = genome.values + noise
    return RealGenome(values=new_values)

# Apply mutation across population
mutation_keys = jr.split(jr.PRNGKey(456), len(real_pop))
mutated_genes = jax.vmap(add_noise, in_axes=(0, 0))(mutation_keys, real_pop.genes)

# Create offspring population (fitness reset to NaN)
offspring_pop = real_pop.spawn_offspring(mutated_genes)

print(f"Offspring fitness (should be NaN): {offspring_pop.fitness[:5]}")
print(f"Original gene[0][0]: {real_pop.genes.values[0, 0]:.4f}")
print(f"Mutated gene[0][0]:  {offspring_pop.genes.values[0, 0]:.4f}")

Offspring fitness (should be NaN): [nan nan nan nan nan]
Original gene[0][0]: -2.7015
Mutated gene[0][0]:  -2.5933


---
## Summary

| Component | Purpose |
|-----------|--------|
| `BinaryGenomeConfig` | Config for bit-strings (`shape` or legacy `length`) |
| `BinaryGenome` | Single binary individual with `values`, `flip_bit()`, `to_int()` |
| `RealGenomeConfig` | Config for real vectors with `bounds` |
| `RealGenome` | Single real individual with `values`, `normalize()`, `magnitude()` |
| `*Population` | SoA container with `genes`, `fitness`, slicing, `distance_matrix()` |

All components are JAX-compatible: use `jax.vmap` for batching and `jax.jit` for compilation.